# Sionna SYS

> 5G NR packet error rates from Sionna SYS: LDPC tables, EESM over the subcarriers, link adaptation.

In [ ]:
#| default_exp sionna_sys

In [ ]:
#| hide
from nbdev.showdoc import *

The packet error rates of 5G NR, from [Sionna SYS](https://nvlabs.github.io/sionna/sys/index.html): LDPC-coded
transport blocks, with error rates from Sionna's calibrated tables instead of an analytic formula. It is the error
model of a `SystemLevelChannel`:

```python
channel = SystemLevelChannel(fading=..., noise_power=..., per_model=SionnaErrorModel())
```

It needs Sionna (PyTorch): `pip install 'comm-core[sionna]'`.

**How a transmission becomes a 5G NR transport block:**

1. The message is one transport block of `size_bits`.
2. An MCS (modulation and coding scheme) `m` has a modulation order `Q_m` and a code rate `R_m`. The message takes
   the fewest resource elements (one subcarrier for one OFDM symbol each) whose NR transport block holds it, at
   an effective code rate of at most 0.95, including the CRCs (`nr_allocation`).
3. The SINR at each subcarrier gives one *effective* SINR, by EESM. Faded subcarriers count for more than their
   share of the mean.
4. Sionna's tables give the transport block error rate (TBLER) of that effective SINR, MCS and block size: the
   packet error rate.

**The MCS:** link adaptation picks, for each transmission, the highest MCS whose TBLER is at most `bler_target`.
That's the rule of Sionna's `InnerLoopLinkAdaptation`, with each MCS given the resource elements the message needs
at it. Alternatively, `mcs_index` fixes one MCS.

**The results** add `mcs`, `sinr_eff_db` and `resource_elements` to their metadata. The latency is
`resource elements / bandwidth`.

**Limits:**

* The tables are 5G NR's (its LDPC codes and MCS tables), not Wi-Fi's.
* Below the tables' range (about −5 dB), packets are lost.
* There is no HARQ soft combining. A protocol's `max_retries` resends lost messages as new transmissions.

In [ ]:
#| export
from __future__ import annotations

import math
from typing import Sequence

import numpy as np

from comm_core.system import ErrorModel

In [ ]:
#| export
MAX_CODERATE = 0.95                                        # NR: a UE may skip decoding above this effective code rate


def nr_allocation(
    num_bits: int, # Payload size
    modulation_order: int, # Bits per QAM symbol of the MCS
    coderate: float, # Code rate of the MCS
) -> tuple:
    "(resource elements, TB size): the fewest resource elements whose NR transport block holds `num_bits`."
    import torch
    from sionna.phy.nr.utils import calculate_tb_size
    num_bits = max(int(num_bits), 1)
    first = max(1, math.ceil(num_bits / (modulation_order * coderate)))
    for lo in range(first, first + 100_000, 2048):         # candidates in chunks; usually the first ones fit
        re = torch.arange(lo, lo + 2048)
        tb, _, num_cb, tb_crc, cb_crc = calculate_tb_size(
            modulation_order=torch.full_like(re, modulation_order), target_coderate=torch.full(re.shape, float(coderate)),
            num_coded_bits=re * modulation_order, return_cw_length=False, verbose=False)
        rate = (tb + tb_crc + num_cb * cb_crc) / (re * modulation_order)
        fits = (tb >= num_bits) & (rate <= MAX_CODERATE)
        if fits.any():
            i = int(torch.nonzero(fits)[0])
            return int(re[i]), int(tb[i])
    raise ValueError(f"no NR allocation for {num_bits} bits")


class SionnaErrorModel(ErrorModel):
    "5G NR PHY abstraction of Sionna SYS: EESM over the subcarriers, LDPC BLER tables, an MCS per transmission."

    def __init__(
        self,
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation, the highest MCS meeting `bler_target`
        bler_target: float = 0.1, # Link adaptation's target TBLER
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
        mcs_indices: Sequence[int] | None = None, # Link adaptation's candidate MCSs; None: all of the table's
    ):
        import torch
        from sionna.phy.nr.utils import MCSDecoderNR
        from sionna.sys import EESM, PHYAbstraction
        self._torch = torch
        self.phy = PHYAbstraction(precision=precision, device=device)
        self.eesm = EESM(precision=precision, device=device)
        self.device = self.phy.device
        self.dtype = torch.float64 if precision == 'double' else torch.float32
        self.bler_target, self.mcs_table_index, self.mcs_category = bler_target, mcs_table_index, mcs_category
        decoder = MCSDecoderNR(device=self.device)
        valid = []
        for m in range(32):                                # the MCS indices of this table
            try:
                decoder(torch.tensor([m], dtype=torch.int32, device=self.device), mcs_table_index=mcs_table_index,
                        mcs_category=mcs_category, check_validity=True)
                valid.append(m)
            except ValueError:
                pass
        if mcs_indices is not None:
            valid = [m for m in valid if m in set(mcs_indices)]
        if mcs_index is not None and mcs_index not in valid:
            raise ValueError(f"mcs_index {mcs_index}: one of {valid} for MCS table {mcs_table_index}")
        self.mcs_index = mcs_index
        self.mcs = torch.tensor(valid if mcs_index is None else [mcs_index], dtype=torch.int32, device=self.device)
        order, rate = decoder(self.mcs, mcs_table_index=mcs_table_index, mcs_category=mcs_category, check_validity=False)
        self.modulation_order, self.coderate = order.cpu().tolist(), rate.cpu().tolist()
        self._allocations = {}                             # (bits, MCS position) -> (resource elements, TB size)

    def allocation(
        self,
        num_bits: int, # Payload size
        mcs: int, # MCS index
    ) -> tuple:
        "(resource elements, TB size) of a payload at an MCS (see `nr_allocation`), cached."
        i = self.mcs.cpu().tolist().index(mcs)
        if (num_bits, i) not in self._allocations:
            self._allocations[num_bits, i] = nr_allocation(num_bits, self.modulation_order[i], self.coderate[i])
        return self._allocations[num_bits, i]

    def __call__(self, sinr, size_bits):
        torch = self._torch
        n, width, M = len(sinr), max(len(s) for s in sinr), len(self.mcs)
        grid = np.zeros((n, width))                        # 0: a subcarrier the transmission does not use
        for k, s in enumerate(sinr):
            grid[k, :len(s)] = np.minimum(s, 1e12)
        s = torch.as_tensor(grid, dtype=self.dtype, device=self.device)[:, None, :, None, None].expand(n, 1, width, M, 1)
        mcs = self.mcs[None].expand(n, M)
        sinr_eff = self.eesm(s, mcs, mcs_table_index=self.mcs_table_index, mcs_category=self.mcs_category)
        mcs_list = self.mcs.cpu().tolist()
        re = torch.as_tensor([[self.allocation(max(int(b), 1), m)[0] for m in mcs_list] for b in size_bits],
                             dtype=torch.int32, device=self.device)
        tbler = self.phy(mcs, sinr_eff=sinr_eff, num_allocated_re=re, mcs_table_index=self.mcs_table_index,
                         mcs_category=self.mcs_category, check_mcs_index_validity=False)[3]
        tbler = torch.nan_to_num(tbler, nan=1.0, posinf=1.0).clamp(0, 1)   # no table entry: lost
        if self.mcs_index is None:                          # the highest MCS meeting the target, else the lowest
            ok = tbler <= self.bler_target
            best = torch.where(ok.any(1), (ok * torch.arange(M, device=self.device)).argmax(1),
                               torch.zeros(n, dtype=torch.long, device=self.device))
        else:
            best = torch.zeros(n, dtype=torch.long, device=self.device)
        rows = torch.arange(n, device=self.device)
        per = tbler[rows, best].cpu().numpy().astype(float)
        res = re[rows, best].cpu().numpy().astype(int)
        eff = sinr_eff[rows, best].cpu().numpy().astype(float)
        mcs_used = self.mcs[best].cpu().numpy()
        info = [{'mcs': int(m), 'sinr_eff_db': 10 * math.log10(e) if e > 0 else -math.inf, 'resource_elements': int(r)}
                for m, e, r in zip(mcs_used, eff, res)]
        return per, res, info

In [ ]:
show_doc(SionnaErrorModel.__call__)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_sys.py#L95){target="_blank" style="float:right; font-size:smaller"}

### SionnaErrorModel.__call__

```python
def __call__(
    sinr, size_bits
):
```

*(packet error rates, symbols on the air, extra metadata) of the transmissions.*

In [ ]:
show_doc(nr_allocation)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_sys.py#L23){target="_blank" style="float:right; font-size:smaller"}

### nr_allocation

```python
def nr_allocation(
    num_bits:int, # Payload size
    modulation_order:int, # Bits per QAM symbol of the MCS
    coderate:float, # Code rate of the MCS
)->tuple:
```

*(resource elements, TB size): the fewest resource elements whose NR transport block holds `num_bits`.*

In [ ]:
from comm_core.system import BPSKErrorModel

### Tests

These need Sionna, so they are flagged (`#| sionna`): `nbdev_test --flags sionna` runs them.

The packet error rate of a 1000-bit message over a flat channel, link adaptation choosing the MCS, compared
with uncoded BPSK (`BPSKErrorModel`):

In [ ]:
#| sionna
model = SionnaErrorModel()
snrs = [-6, -3, 0, 3, 6, 10, 15, 20, 25]
per, re, info = model([np.array([10 ** (x / 10)]) for x in snrs], np.full(len(snrs), 1000))
bpsk = BPSKErrorModel()([np.array([10 ** (x / 10)]) for x in snrs], np.full(len(snrs), 1000))[0]
print(' SNR   MCS  resource elements  TBLER   uncoded BPSK')
for x, p, i, b in zip(snrs, per, info, bpsk):
    print(f'{x:4d}  {i["mcs"]:4d}  {i["resource_elements"]:17d}  {p:5.3f}   {b:5.3f}')
assert per[0] == 1.0 and (per[2:] <= 0.1).all()                     # below the tables: lost; above: the target
mcs = [i['mcs'] for i in info]
assert mcs == sorted(mcs) and mcs[-1] > mcs[2] and re[-1] < re[2]      # better channels: faster MCS, fewer REs

 SNR   MCS  resource elements  TBLER   uncoded BPSK
  -6     0               4267  1.000   1.000
  -3     0               4267  1.000   1.000
   0     5               1351  0.000   1.000
   3     9                755  0.039   1.000
   6    13                523  0.097   0.908
  10    17                390  0.005   0.004
  15    22                257  0.012   0.000
  20    27                188  0.020   0.000
  25    27                188  0.020   0.000


A frequency-selective channel: EESM weighs the faded subcarriers, so the effective SINR is below the mean.

In [ ]:
#| sionna
rng = np.random.default_rng(0)
h = (rng.standard_normal(52) + 1j * rng.standard_normal(52)) / np.sqrt(2)     # Rayleigh across 52 subcarriers
fixed = SionnaErrorModel(mcs_index=15)
_, _, (flat,) = fixed([np.array([10 ** 1.5])], np.array([1000]))
_, _, (sel,) = fixed([10 ** 1.5 * np.abs(h) ** 2], np.array([1000]))
print(f"mean SINR 15 dB: effective {flat['sinr_eff_db']:.1f} dB flat, {sel['sinr_eff_db']:.1f} dB frequency-selective")
assert abs(flat['sinr_eff_db'] - 15) < 0.1 and sel['sinr_eff_db'] < flat['sinr_eff_db'] - 1
try:
    SionnaErrorModel(mcs_index=40); raise AssertionError('no error')
except ValueError:
    pass

mean SINR 15 dB: effective 15.0 dB flat, 10.4 dB frequency-selective


In a network, the steps' transmissions go to Sionna in one call. Here, four nodes on a line take turns
(`TDMAProtocol`, each sending its newest round of messages), over a channel ray-traced at anchors with 52 subcarriers. The error model sees the per-subcarrier
SINRs:

In [ ]:
from comm_core import (Network, Node, Message, FullMeshTopology, TDMAProtocol, InterferenceBackend,
                       SystemLevelChannel, RayTracedPathsFading, grid_points)

In [ ]:
#| sionna
C, F = 299_792_458.0, 5.18e9
points = grid_points((0, 0), (30, 0), 0.5, height=1.0)                 # anchors every 50 cm on a 30 m corridor
v = points[None] - points[:, None]                                       # line of sight (tx, rx)
L = np.linalg.norm(v, axis=-1)
same = L == 0
L = np.where(same, 1.0, L)
paths = dict(a=np.where(same, 0, C / F / (4 * np.pi * L) * np.exp(-2j * np.pi * F * L / C))[..., None],
             tau=(L / C)[..., None], u_tx=(v / L[..., None])[..., None, :], u_rx=(-v / L[..., None])[..., None, :])
fading = RayTracedPathsFading(points, points, **paths, frequency=F, subcarriers=(np.arange(52) - 25.5) * 312.5e3)
nodes = ['a', 'b', 'c', 'd']
fading.set_positions({'a': (0.3, 0, 1), 'b': (6.1, 0, 1), 'c': (14.8, 0, 1), 'd': (29.6, 0, 1)})
channel = SystemLevelChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), per_model=SionnaErrorModel(), seed=0)
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes, queue_size=3), InterferenceBackend(channel))
for link in (net.topology.get_link(s, r) for s in nodes for r in nodes if s != r):
    link.tx_power_w, link.bandwidth_hz = 1e-4, 20e6                     # 0.1 mW, 20 MHz
for step in range(4):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, None, size_bits=2000))
    for res in net.step():
        t = res.transmission
        print(f'{t.sender} -> {t.receiver}: SINR {res.snr_db:5.1f} dB, MCS {res.metadata["mcs"]:2d}, '
              f'{res.metadata["resource_elements"]:5d} REs, {1e6 * res.latency:5.1f} µs, delivered {res.success}')
m = net.metrics.summary()
assert m.transmissions == 12 and m.successful >= 10

a -> b: SINR  22.0 dB, MCS 27,   376 REs,  18.8 µs, delivered True
a -> c: SINR  14.0 dB, MCS 21,   555 REs,  27.8 µs, delivered True
a -> d: SINR   7.9 dB, MCS 15,   832 REs,  41.6 µs, delivered True
b -> a: SINR  22.0 dB, MCS 27,   376 REs,  18.8 µs, delivered True
b -> c: SINR  18.5 dB, MCS 26,   391 REs,  19.6 µs, delivered True
b -> d: SINR   9.8 dB, MCS 17,   780 REs,  39.0 µs, delivered True
c -> a: SINR  14.0 dB, MCS 21,   555 REs,  27.8 µs, delivered True
c -> b: SINR  18.5 dB, MCS 26,   391 REs,  19.6 µs, delivered True
c -> d: SINR  13.9 dB, MCS 21,   555 REs,  27.8 µs, delivered True
d -> a: SINR   7.9 dB, MCS 15,   832 REs,  41.6 µs, delivered True
d -> b: SINR   9.8 dB, MCS 17,   780 REs,  39.0 µs, delivered True
d -> c: SINR  13.9 dB, MCS 21,   555 REs,  27.8 µs, delivered True


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()